# Import

In [146]:
import os
import boto3
import pickle
import warnings
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import xgboost as xgb
import sklearn
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
	OneHotEncoder,
	OrdinalEncoder,
	StandardScaler,
	MinMaxScaler,
	PowerTransformer,
	FunctionTransformer
)

from feature_engine.outliers import Winsorizer
from feature_engine.datetime import DatetimeFeatures
from feature_engine.selection import SelectBySingleFeaturePerformance
from feature_engine.encoding import (
	RareLabelEncoder,
	MeanEncoder,
	CountFrequencyEncoder
)

import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.tuner import (
    IntegerParameter,
    ContinuousParameter,
    HyperparameterTuner
)

In [147]:
!pip install xgboost==1.7.3

In [148]:
# 2. Display Settings

In [149]:
pd.set_option("display.max_columns",None)
sklearn.set_config(transform_output="pandas")
warnings.filterwarnings("ignore")

# 3. Read Datasets

In [150]:
train = pd.read_csv("train.csv")
train

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 train = pd.read_csv("train.csv")                                                             │
│   2 train                                                                                        │
│   3                                                                                              │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/pandas/io/parsers/readers.py: │
│ 1026 in read_csv                                                                                 │
│                                                                                                  │
│   1023 │   )                                                                                     │
│   1024 │   kwds.update(kwds_defaults)                                                            │
│   1025 │                                                                                         │
│ ❱ 1026 │   return _read(filepath_or_buffer, kwds)                                                │
│   1027                                                                                           │
│   1028                                                                                           │
│   1029 # iterator=True -> TextFileReader                                                         │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/pandas/io/parsers/readers.py: │
│ 620 in _read                                                                                     │
│                                                                                                  │
│    617 │   _validate_names(kwds.get("names", None))                                              │
│    618 │                                                                                         │
│    619 │   # Create the parser.                                                                  │
│ ❱  620 │   parser = TextFileReader(filepath_or_buffer, **kwds)                                   │
│    621 │                                                                                         │
│    622 │   if chunksize or iterator:                                                             │
│    623 │   │   return parser                                                                     │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/pandas/io/parsers/readers.py: │
│ 1620 in __init__                                                                                 │
│                                                                                                  │
│   1617 │   │   │   self.options["has_index_names"] = kwds["has_index_names"]                     │
│   1618 │   │                                                                                     │
│   1619 │   │   self.handles: IOHandles | None = None                                             │
│ ❱ 1620 │   │   self._engine = self._make_engine(f, self.engine)                                  │
│   1621 │                                                                                         │
│   1622 │   def close(self) -> None:                                                              │
│   1623 │   │   if self.handles is not None:                                                      │
│                                                            

In [ ]:
val = pd.read_csv("val.csv")
val

In [ ]:
test = pd.read_csv("test.csv")
test

# 4. Preprocessing Operations

In [151]:
# airline
air_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("grouper", RareLabelEncoder(tol=0.1, replace_with="Other", n_categories=2)),
    ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
])

#doj
feature_to_extract = ["month", "week", "day_of_week", "day_of_year"]

doj_transformer = Pipeline(steps=[
    ("dt", DatetimeFeatures(features_to_extract=feature_to_extract, yearfirst=True, format="mixed")),
    ("scaler", MinMaxScaler())
])

# source & destination
location_pipe1 = Pipeline(steps=[
    ("grouper", RareLabelEncoder(tol=0.1, replace_with="Other", n_categories=2)),
    ("encoder", MeanEncoder()),
    ("scaler", PowerTransformer())
])

def is_north(X):
    columns = X.columns.to_list()
    north_cities = ["Delhi", "Kolkata", "Mumbai", "New Delhi"]
    return (
        X
        .assign(**{
            f"{col}_is_north": X.loc[:, col].isin(north_cities).astype(int)
            for col in columns
        })
        .drop(columns=columns)
    )

location_transformer = FeatureUnion(transformer_list=[
    ("part1", location_pipe1),
    ("part2", FunctionTransformer(func=is_north))
])

# dep_time & arrival_time
time_pipe1 = Pipeline(steps=[
    ("dt", DatetimeFeatures(features_to_extract=["hour", "minute"])),
    ("scaler", MinMaxScaler())
])

def part_of_day(X, morning=4, noon=12, eve=16, night=20):
    columns = X.columns.to_list()
    X_temp = X.assign(**{
        col: pd.to_datetime(X.loc[:, col]).dt.hour
        for col in columns
    })

    return (
        X_temp
        .assign(**{
            f"{col}_part_of_day": np.select(
                [X_temp.loc[:, col].between(morning, noon, inclusive="left"),
                 X_temp.loc[:, col].between(noon, eve, inclusive="left"),
                 X_temp.loc[:, col].between(eve, night, inclusive="left")],
                ["morning", "afternoon", "evening"],
                default="night"
            )
            for col in columns
        })
        .drop(columns=columns)
    )

time_pipe2 = Pipeline(steps=[
    ("part", FunctionTransformer(func=part_of_day)),
    ("encoder", CountFrequencyEncoder()),
    ("scaler", MinMaxScaler())
])

time_transformer = FeatureUnion(transformer_list=[
    ("part1", time_pipe1),
    ("part2", time_pipe2)
])

# duration
class RBFPercentileSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, variables=None, percentiles=[0.25, 0.5, 0.75], gamma=0.1):
        self.variables = variables
        self.percentiles = percentiles
        self.gamma = gamma


    def fit(self, X, y=None):
        if not self.variables:
            self.variables = X.select_dtypes(include="number").columns.to_list()

        self.reference_values_ = {
            col: (
                X
                .loc[:, col]
                .quantile(self.percentiles)
                .values
                .reshape(-1, 1)
            )
            for col in self.variables
        }

        return self


    def transform(self, X):
        objects = []
        for col in self.variables:
            columns = [f"{col}_rbf_{int(percentile * 100)}" for percentile in self.percentiles]
            obj = pd.DataFrame(
                data=rbf_kernel(X.loc[:, [col]], Y=self.reference_values_[col], gamma=self.gamma),
                columns=columns
            )
            objects.append(obj)
        return pd.concat(objects, axis=1)
    

def duration_category(X, short=180, med=400):
    return (
        X
        .assign(duration_cat=np.select([X.duration.lt(short),
                                    X.duration.between(short, med, inclusive="left")],
                                    ["short", "medium"],
                                    default="long"))
        .drop(columns="duration")
    )

def is_over(X, value=1000):
    return (
        X
        .assign(**{
            f"duration_over_{value}": X.duration.ge(value).astype(int)
        })
        .drop(columns="duration")
    )

duration_pipe1 = Pipeline(steps=[
    ("rbf", RBFPercentileSimilarity()),
    ("scaler", PowerTransformer())
])

duration_pipe2 = Pipeline(steps=[
    ("cat", FunctionTransformer(func=duration_category)),
    ("encoder", OrdinalEncoder(categories=[["short", "medium", "long"]]))
])

duration_union = FeatureUnion(transformer_list=[
    ("part1", duration_pipe1),
    ("part2", duration_pipe2),
    ("part3", FunctionTransformer(func=is_over)),
    ("part4", StandardScaler())
])

duration_transformer = Pipeline(steps=[
    ("outliers", Winsorizer(capping_method="iqr", fold=1.5)),
    ("imputer", SimpleImputer(strategy="median")),
    ("union", duration_union)
])

# total_stops
def is_direct(X):
    return X.assign(is_direct_flight=X.total_stops.eq(0).astype(int))


total_stops_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("", FunctionTransformer(func=is_direct))
])

# additional_info
info_pipe1 = Pipeline(steps=[
    ("group", RareLabelEncoder(tol=0.1, n_categories=2, replace_with="Other")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

def have_info(X):
    return X.assign(additional_info=X.additional_info.ne("No Info").astype(int))

info_union = FeatureUnion(transformer_list=[
("part1", info_pipe1),
("part2", FunctionTransformer(func=have_info))
])

info_transformer = Pipeline(steps=[
("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
("union", info_union)
])

# column transformer
column_transformer = ColumnTransformer(transformers=[
("air", air_transformer, ["airline"]),
("doj", doj_transformer, ["date_of_journey"]),
("location", location_transformer, ["source", 'destination']),
("time", time_transformer, ["dep_time", "arrival_time"]),
("dur", duration_transformer, ["duration"]),
("stops", total_stops_transformer, ["total_stops"]),
("info", info_transformer, ["additional_info"])
], remainder="passthrough")

# feature selector
estimator = RandomForestRegressor(n_estimators=10, max_depth=3, random_state=42)

selector = SelectBySingleFeaturePerformance(
estimator=estimator,
scoring="r2",
threshold=0.1
) 

# preprocessor
preprocessor = Pipeline(steps=[
("ct", column_transformer),
("selector", selector)
])

In [152]:
preprocessor.fit(
    train.drop(columns="price"),
    train.price.copy()
)

,steps,"[('ct', ...), ('selector', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('air', ...), ('doj', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [153]:
preprocessor.transform(train.drop(columns="price"))

,air__airline_Indigo,air__airline_Jet Airways,air__airline_Other,doj__date_of_journey_week,doj__date_of_journey_day_of_year,location__source,location__destination,time__arrival_time_hour,dur__duration_rbf_25,dur__duration_cat,dur__duration_over_1000,dur__duration,stops__total_stops,stops__is_direct_flight
0,0.0,0.0,0.0,0.764706,0.737288,1.000723,1.000621,0.913043,-0.356222,2.0,0,-0.407958,1.0,0
1,1.0,0.0,0.0,0.235294,0.220339,1.000723,1.000621,0.913043,-0.356222,2.0,0,0.465097,1.0,0
2,0.0,1.0,0.0,0.588235,0.584746,-0.097370,-0.096531,0.826087,-0.356222,2.0,0,0.346044,1.0,0
3,0.0,0.0,0.0,0.588235,0.584746,-1.778504,-1.044956,0.565217,-0.356222,0.0,0,-0.973460,0.0,1
4,1.0,0.0,0.0,0.058824,0.067797,-1.072220,-1.044956,0.000000,2.427136,0.0,0,-0.894091,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,0.0,0.0,0.0,1.000000,1.000000,1.000723,1.000621,0.043478,-0.356222,2.0,0,0.603992,1.0,0
636,0.0,1.0,0.0,0.823529,0.822034,-0.097370,-0.096531,0.434783,-0.356222,2.0,0,0.484939,1.0,0
637,0.0,0.0,0.0,0.058824,0.067797,1.000723,1.000621,0.913043,-0.356222,2.0,0,0.306360,1.0,0
638,0.0,0.0,0.0,0.058824,0.067797,1.000723,1.000621,0.826087,-0.356222,2.0,0,0.217070,1.0,0


# 4. Preprocess Data and Upload to Bucket

In [154]:
BUCKET_NAME = "sagemaker-flights-bucketss"
DATA_PREFIX = "data"

In [155]:
def get_file_name(name):
    return f"{name}-pre.csv"

In [156]:
def export_data(data, name, pre):
    # split data into X and y subsets
    X = data.drop(columns="price")
    y = data.price.copy()
    
    # transformation
    X_pre = pre.transform(X)
    
    # exporting
    file_name = get_file_name(name)
    (
        y
        .to_frame()
        .join(X_pre)
        .to_csv(file_name, index=False, header=False)
    )

In [157]:
def upload_to_bucket(name):
    file_name = get_file_name(name)
    (
        boto3
        .Session()
        .resource("s3")
        .Bucket(BUCKET_NAME)
        .Object(os.path.join(DATA_PREFIX, f"{name}/{name}.csv"))
        .upload_file(file_name)
    )

In [158]:
def export_and_upload_bucket(data, name, pre):
    export_data(data, name, pre)
    upload_to_bucket(name)

In [159]:
export_and_upload_bucket(train, "train", preprocessor)

In [160]:
export_and_upload_bucket(val, "val", preprocessor)

In [161]:
export_and_upload_bucket(test, "test", preprocessor)

# 5. Model and Hyperparameter Tuning Set-up

In [162]:
session = sagemaker.Session()
region_name = session.boto_region_name

In [163]:
output_path = f"s3://{BUCKET_NAME}/model/output" 

In [164]:
model = Estimator(
    image_uri=sagemaker.image_uris.retrieve("xgboost", region_name, "1.2-1"),
    role=sagemaker.get_execution_role(),
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size=5,
    output_path=output_path,
    use_spot_instances=True,
    max_run=300,
    max_wait=600,
    sagemaker_session=session
)

In [165]:
model.set_hyperparameters(
    objective="reg:linear",
    num_round=10,
    eta=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    alpha=0.1
)

In [166]:
hyperparameter_ranges = {
    "eta": ContinuousParameter(0.05, 0.2),
    "alpha": ContinuousParameter(0, 1),
    "max_depth": IntegerParameter(3, 5)
}

In [167]:
tuner = HyperparameterTuner(
    estimator=model,
    objective_metric_name="validation:rmse",
    hyperparameter_ranges=hyperparameter_ranges,
    strategy="Bayesian",
    objective_type="Minimize"
)

# 6. Data Channels

In [168]:
def get_data_channel(name):
    bucket_path = f"s3://{BUCKET_NAME}/{DATA_PREFIX}/{name}"
    return TrainingInput(bucket_path, content_type="csv")

In [169]:
train_data_channel = get_data_channel("train")
train_data_channel

In [170]:
val_data_channel = get_data_channel("val")

In [171]:
data_channels = {
    "train": train_data_channel,
    "validation": val_data_channel
}

# 7. Train and Tune the Model

In [ ]:
tuner.fit(data_channels)

No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config
No finished training job found associated with this estimator. Please make sure this estimator is only used for building workflow config


.....

# 8. Model Evaluation

In [140]:

# 1. Load the old pickle file (which works in 1.7.3)
with open("xgboost-model", "rb") as f:
    old_pickle_model = pickle.load(f)

# 2. Extract the underlying booster and save it natively as JSON
# Since it predicts price, we use XGBRegressor
if isinstance(old_pickle_model, xgb.XGBRegressor):
    old_pickle_model.save_model("modern_xgboost_model.json")
    print("Success! Model converted to modern_xgboost_model.json")
else:
    # If it was saved as a pure Booster instead of the Sklearn wrapper
    old_pickle_model.save_model("modern_xgboost_model.json")
    print("Success! Booster converted to modern_xgboost_model.json")

[14:38:56] WARNING: ../src/objective/regression_obj.cu:213: reg:linear is now deprecated in favor of reg:squarederror.
[14:38:56] WARNING: ../src/learner.cc:553: 
  If you are loading a serialized model (like pickle in Python, RDS in R) generated by
  older XGBoost, please export the model by calling `Booster.save_model` from that version
  first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/latest/tutorials/saving_model.html

  for more details about differences between saving model and serializing.

Success! Booster converted to modern_xgboost_model.json


In [141]:
print("Model load ho gaya!")
print(type(best_model))
print(best_model)

Model load ho gaya!
<class 'xgboost.sklearn.XGBRegressor'>
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, gpu_id=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             n_estimators=100, n_jobs=None, num_parallel_tree=None,
             predictor=None, random_state=None, ...)


In [142]:
best_model = xgb.XGBRegressor()

# 2. Load the JSON file directly (Latest XGBoost loves this format)
best_model.load_model("modern_xgboost_model.json")
print("Model loaded successfully on the LATEST XGBoost version!")

# 3. Your evaluation function will now work perfectly
def evaluate_model(split):
    if split == "train":
        data = train
    elif split == "val":
        data = val
    else:
        data = test

    X = data.drop(columns="price")
    y = data["price"].copy()

    X_transformed = preprocessor.transform(X)

    if hasattr(X_transformed, 'values'):
        X_transformed = X_transformed.values
        
    # Note: Modern XGBoost Scikit-Learn wrappers can predict 
    # directly from numpy arrays, no DMatrix conversion required!
    pred = best_model.predict(X_transformed) 
    
    r2 = r2_score(y, pred)
    print(f"--- {split.upper()} PERFORMANCE (LATEST VERSION) ---")
    print(f"R2 Score: {r2:.4f}")
    return r2

# Test it out
evaluate_model("train")

Model loaded successfully on the LATEST XGBoost version!
--- TRAIN PERFORMANCE (LATEST VERSION) ---
R2 Score: 0.1959


0.19591236114501953

In [143]:
evaluate_model("train")

--- TRAIN PERFORMANCE (LATEST VERSION) ---
R2 Score: 0.1959


0.19591236114501953

In [144]:
evaluate_model("val")

--- VAL PERFORMANCE (LATEST VERSION) ---
R2 Score: 0.1066


0.10659319162368774

In [145]:
evaluate_model("test")

--- TEST PERFORMANCE (LATEST VERSION) ---
R2 Score: 0.2496


0.2495517134666443